In [6]:
import re, html, pandas as pd, spacy

df = pd.read_csv("../data/processed/clean.csv")
df = df.dropna(subset=["clean"])
print(df.shape)                       # (79788, 7)

splitter = spacy.blank("en")
splitter.add_pipe("sentencizer")

ASPECTS = {
    "pacing":     ["pace", "pacing", "paced", "slow", "drag", "dragged", "drags", "rushed"],
    "characters": ["character", "characters", "protagonist", "heroine", "narrator"],
    "plot":       ["plot", "storyline", "story", "twist", "twists"],
    "ending":     ["ending", "endings", "finale", "epilogue"],
    "writing":    ["writing", "prose", "style", "narration", "writer","well written", "well-written", "poorly written","badly written", "beautifully written"],
    "emotion":    ["emotional", "emotions", "heartbreaking", "touching", "tears", "cried"],
}
ASPECT_RE = {a: re.compile(r"\b(" + "|".join(kws) + r")\b") for a, kws in ASPECTS.items()}

(79788, 7)


In [7]:
sub = df.sample(10_000, random_state=42).reset_index(drop=True)
titles = sub["title"].tolist()
labels = sub["label"].tolist()

rows = []
for i, doc in enumerate(splitter.pipe(sub["text"].map(html.unescape), batch_size=200)):
    for s in doc.sents:
        sent = s.text.strip()
        if len(sent.split()) < 4:
            continue
        low = sent.lower()
        for aspect, rx in ASPECT_RE.items():
            if rx.search(low):
                rows.append((i, titles[i], labels[i], aspect, sent))

sents = pd.DataFrame(rows, columns=["review_id", "title", "review_label", "aspect", "sentence"])
print(len(sents))
print(sents["aspect"].value_counts())

11595
aspect
plot          4477
characters    3202
writing       2540
emotion        522
pacing         431
ending         423
Name: count, dtype: int64


In [3]:
for a in ASPECTS:
    print(a.upper())
    for t in sents[sents["aspect"] == a]["sentence"].sample(3, random_state=1):
        print("  -", t[:200])

PACING
  - By the end of this book, it struck me that it's very much like a lot of historical and military fiction: decent at getting across the salient facts, decent at putting the reader in the middle of the a
  - Fast paced action revolving around a real weapon system.
  - He has the skill to smoothly draw the reader into an entertaining story, and then to slowly put the reader off-guard until you find yourself wrapped up in the middle of a very dark narrative.
CHARACTERS
  - I can't remember the last time I read a book that began with a two-page list of characters!
  - So what were my issues?In Northanger Abbey, the heroine is also fairly naive and kind of creates the problems in her life.
  - I have begun to make myself another character in the book!
PLOT
  - THIS STORY WAS VERY CAPTIVATING AND EASY TO READ,UNDERSTAND AND FOLLOW.I'VE ALWAYS LOVED THIS STORY AND NOW I DO EVEN MORE THAT BEFORE.HOWERVER, THE ENDING WAS QUITE DIFFERENT THEN I REMEMBER.
  - Harold Bell Wright weaves an

In [5]:
from transformers import pipeline
import time

clf = pipeline("text-classification",
               model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
               device=-1)

t0 = time.time()
clf(sents["sentence"].head(200).tolist(), batch_size=32, truncation=True, max_length=128)
secs = time.time() - t0
print(round(secs, 1), "seconds for 200 sentences")
print("estimated minutes for all:", round(secs / 200 * len(sents) / 60, 1))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1030.28it/s]


10.7 seconds for 200 sentences
estimated minutes for all: 11.5


In [8]:
out = clf(sents["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
sents["positive"] = [o["label"] == "POSITIVE" for o in out]
sents.to_csv("../data/processed/aspect_sentences.csv", index=False)
print(sents["positive"].mean())

0.6930573523070289


In [9]:
summary = sents.groupby("aspect").agg(sentences=("positive", "size"),
                                      pct_positive=("positive", "mean"))
summary["pct_positive"] = (summary["pct_positive"] * 100).round(1)
print(summary)

# how often each aspect appears in negative vs positive reviews
n_by_label = sub["label"].value_counts()
mention = (sents.drop_duplicates(["review_id", "aspect"])
                .groupby(["aspect", "review_label"]).size().unstack(fill_value=0))
mention_pct = (mention / n_by_label * 100).round(1)
mention_pct.columns = ["% of negative reviews", "% of positive reviews"]
print(mention_pct)

            sentences  pct_positive
aspect                             
characters       3202          66.6
emotion           522          75.5
ending            423          54.4
pacing            431          54.5
plot             4477          70.9
writing          2540          73.7
            % of negative reviews  % of positive reviews
aspect                                                  
characters                   24.4                   19.8
emotion                       1.9                    4.9
ending                        4.5                    3.7
pacing                        4.9                    3.8
plot                         27.6                   27.2
writing                      23.3                   18.7


In [10]:
from collections import Counter

counts = Counter()
for chunk in pd.read_csv("../data/raw/archive/Books_rating.csv",
                         usecols=["Title"], chunksize=500_000):
    counts.update(chunk["Title"].dropna())

top = pd.Series(counts).sort_values(ascending=False).head(30)
print(top)

The Hobbit                                                          22023
Pride and Prejudice                                                 20371
Atlas Shrugged                                                      12513
Wuthering Heights                                                   10780
The Giver                                                            7644
Great Expectations                                                   7421
Harry Potter and The Sorcerer's Stone                                6796
Of Mice and Men                                                      6728
Brave New World                                                      6312
Mere Christianity                                                    6053
The Picture of Dorian Gray                                           5883
Persuasion                                                           5498
The Great Gatsby                                                     5291
Fahrenheit 451                        

In [11]:
chosen = ["Pride and Prejudice", "Atlas Shrugged", "Wuthering Heights",
          "The Giver", "Mere Christianity"]

parts = []
for chunk in pd.read_csv("../data/raw/archive/Books_rating.csv",
                         usecols=["Title", "review/score", "review/text"],
                         chunksize=500_000):
    parts.append(chunk[chunk["Title"].isin(chosen)])

books = pd.concat(parts).dropna(subset=["review/text"])
books = books.rename(columns={"review/score": "score", "review/text": "text", "Title": "title"})

print("before dedup:", len(books))
books = books.drop_duplicates(subset=["title", "text"])
print("after dedup: ", len(books))
print("duplicate texts across titles:", books["text"].duplicated().sum())
print(books["title"].value_counts())

books.to_csv("../data/processed/book_reviews.csv", index=False)

before dedup: 57361
after dedup:  10918
duplicate texts across titles: 7
title
The Giver              3739
Atlas Shrugged         3116
Pride and Prejudice    2004
Wuthering Heights      1058
Mere Christianity      1001
Name: count, dtype: int64


In [6]:
books = pd.read_csv("../data/processed/book_reviews.csv")
books = pd.concat([g.sample(min(len(g), 1500), random_state=42)
                   for _, g in books.groupby("title")], ignore_index=True)
print(books["title"].value_counts())

titles = books["title"].tolist()
rows = []
for i, doc in enumerate(splitter.pipe(books["text"].map(html.unescape), batch_size=200)):
    for s in doc.sents:
        sent = s.text.strip()
        if len(sent.split()) < 4:
            continue
        low = sent.lower()
        for aspect, rx in ASPECT_RE.items():
            if rx.search(low):
                rows.append((i, titles[i], aspect, sent))

bs = pd.DataFrame(rows, columns=["review_id", "title", "aspect", "sentence"])
print(len(bs))
print(bs.groupby(["title", "aspect"]).size().unstack(fill_value=0))

title
Atlas Shrugged         1500
Pride and Prejudice    1500
The Giver              1500
Wuthering Heights      1058
Mere Christianity      1001
Name: count, dtype: int64
9358
aspect               characters  emotion  ending  pacing  plot  writing
title                                                                  
Atlas Shrugged              812       57      27      53   842      389
Mere Christianity            16       12       2       8    24      238
Pride and Prejudice        1003       56      49      68   936      323
The Giver                   328      169     369      58   746      199
Wuthering Heights           955      140      79      53  1042      305


In [3]:
out = clf(bs["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
bs["positive"] = [o["label"] == "POSITIVE" for o in out]
bs.to_csv("../data/processed/book_aspect_sentences.csv", index=False)

NameError: name 'clf' is not defined

In [ ]:
out = clf(bs["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
bs["positive"] = [o["label"] == "POSITIVE" for o in out]
bs.to_csv("../data/processed/book_aspect_sentences.csv", index=False)

: 

In [1]:
out = clf(bs["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
bs["positive"] = [o["label"] == "POSITIVE" for o in out]
bs.to_csv("../data/processed/book_aspect_sentences.csv", index=False)

NameError: name 'clf' is not defined

In [2]:
out = clf(bs["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
bs["positive"] = [o["label"] == "POSITIVE" for o in out]
bs.to_csv("../data/processed/book_aspect_sentences.csv", index=False)

NameError: name 'clf' is not defined

In [3]:
for name in ["pd", "html", "splitter", "ASPECT_RE", "clf", "books", "bs"]:
    print(name, name in globals())

pd False
html False
splitter False
ASPECT_RE False
clf False
books False
bs False


In [4]:
import re, html, pandas as pd, spacy
from transformers import pipeline

splitter = spacy.blank("en")
splitter.add_pipe("sentencizer")

ASPECTS = {
    "pacing":     ["pace", "pacing", "paced", "slow", "drag", "dragged", "drags", "rushed"],
    "characters": ["character", "characters", "protagonist", "heroine", "narrator"],
    "plot":       ["plot", "storyline", "story", "twist", "twists"],
    "ending":     ["ending", "endings", "finale", "epilogue"],
    "writing":    ["writing", "prose", "style", "narration", "writer",
                   "well written", "well-written", "poorly written",
                   "badly written", "beautifully written"],
    "emotion":    ["emotional", "emotions", "heartbreaking", "touching", "tears", "cried"],
}
ASPECT_RE = {a: re.compile(r"\b(" + "|".join(kws) + r")\b") for a, kws in ASPECTS.items()}

clf = pipeline("text-classification",
               model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
               device=-1)
print("ready")

c:\venvs\book-nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 352.19it/s]


ready


In [7]:
for name in ["books", "bs"]:
    print(name, name in globals())

books True
bs True


In [8]:
out = clf(bs["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
bs["positive"] = [o["label"] == "POSITIVE" for o in out]
bs.to_csv("../data/processed/book_aspect_sentences.csv", index=False)

In [9]:
t = bs.groupby(["title", "aspect"])["positive"].agg(["mean", "size"]).reset_index()
t["pct_positive"] = (t["mean"] * 100).round(0)
print(t.pivot(index="title", columns="aspect", values="pct_positive"))
print(t.pivot(index="title", columns="aspect", values="size"))

aspect               characters  emotion  ending  pacing  plot  writing
title                                                                  
Atlas Shrugged             49.0     67.0    41.0    43.0  64.0     54.0
Mere Christianity          81.0     25.0    50.0    88.0  67.0     81.0
Pride and Prejudice        83.0     77.0    76.0    50.0  81.0     76.0
The Giver                  77.0     70.0    41.0    45.0  73.0     80.0
Wuthering Heights          56.0     79.0    54.0    51.0  67.0     68.0
aspect               characters  emotion  ending  pacing  plot  writing
title                                                                  
Atlas Shrugged              812       57      27      53   842      389
Mere Christianity            16       12       2       8    24      238
Pride and Prejudice        1003       56      49      68   936      323
The Giver                   328      169     369      58   746      199
Wuthering Heights           955      140      79      53  1042  

In [10]:
MIN_N = 30

def verdict(p, n):
    if n < MIN_N:
        return "not enough data"
    if p >= 70:
        return "mostly praised"
    if p >= 55:
        return "mixed reactions"
    return "often criticized"

def summarize_book(title):
    rows = t[t["title"] == title].sort_values("pct_positive", ascending=False)
    lines = []
    for _, r in rows.iterrows():
        v = verdict(r["pct_positive"], r["size"])
        note = " (low confidence)" if MIN_N <= r["size"] < 100 else ""
        lines.append(f"{r['aspect']}: {v}{note} ({int(r['pct_positive'])}% positive, {int(r['size'])} sentences)")
    return "\n".join(lines)

for title in t["title"].unique():
    print(title)
    print(summarize_book(title))
    print()

Atlas Shrugged
emotion: mixed reactions (low confidence) (67% positive, 57 sentences)
plot: mixed reactions (64% positive, 842 sentences)
writing: often criticized (54% positive, 389 sentences)
characters: often criticized (49% positive, 812 sentences)
pacing: often criticized (low confidence) (43% positive, 53 sentences)
ending: not enough data (41% positive, 27 sentences)

Mere Christianity
pacing: not enough data (88% positive, 8 sentences)
characters: not enough data (81% positive, 16 sentences)
writing: mostly praised (81% positive, 238 sentences)
plot: not enough data (67% positive, 24 sentences)
ending: not enough data (50% positive, 2 sentences)
emotion: not enough data (25% positive, 12 sentences)

Pride and Prejudice
characters: mostly praised (83% positive, 1003 sentences)
plot: mostly praised (81% positive, 936 sentences)
emotion: mostly praised (low confidence) (77% positive, 56 sentences)
ending: mostly praised (low confidence) (76% positive, 49 sentences)
writing: mostly

In [11]:
def examples(title, aspect, positive, k=4):
    s = bs[(bs["title"] == title) & (bs["aspect"] == aspect) & (bs["positive"] == positive)]
    for x in s["sentence"].sample(min(k, len(s)), random_state=1):
        print("-", x[:220])

print("THE GIVER, ending, scored NEGATIVE:")
examples("The Giver", "ending", False)
print("\nATLAS SHRUGGED, characters, scored NEGATIVE:")
examples("Atlas Shrugged", "characters", False)
print("\nPRIDE AND PREJUDICE, characters, scored POSITIVE:")
examples("Pride and Prejudice", "characters", True)

THE GIVER, ending, scored NEGATIVE:
- The ending was okay but that was not the way that I wanted it to end.
- The ending was very incomplete.
- The ending was ambiguous as you're not sure if what the boy is experiencing is a memory or if it is real.
- None of it was given any thought or explanation and it drove me crazy!I also HATED the ending.

ATLAS SHRUGGED, characters, scored NEGATIVE:
- This is annoying and condescending, since the listener is not given the opportunity to develop his own opinions about the characters.
- Her treatment of evil is ham-fisted with her characters nearly caricatures who evolve into straw men easily knocked down by the equally caricature like heroes.
- Imagine what would happen if the people in Washington had the same character and honor as just some of the founding fathers.
- The main character Dagny Taggart will enter a normal setting like a diner and some homeless person will begin talking about how seciety is insane or evil.

PRIDE AND PREJUDICE, cha

In [12]:
import os
bs = pd.read_csv("../data/processed/book_aspect_sentences.csv")

t = bs.groupby(["title", "aspect"])["positive"].agg(["mean", "size"]).reset_index()
t["pct_positive"] = (t["mean"] * 100).round(0)

os.makedirs("../app/results", exist_ok=True)
t[["title", "aspect", "pct_positive", "size"]].rename(columns={"size": "n_sentences"}) \
    .to_csv("../app/results/book_aspects.csv", index=False)

ex = []
for (title, aspect, pos), g in bs.groupby(["title", "aspect", "positive"]):
    g = g[g["sentence"].str.split().str.len().between(6, 40)]
    for s in g["sentence"].sample(min(len(g), 3), random_state=1):
        ex.append((title, aspect, pos, s))
pd.DataFrame(ex, columns=["title", "aspect", "positive", "sentence"]) \
    .to_csv("../app/results/book_examples.csv", index=False)
print(len(t), len(ex))

30 172
